In [1]:
import pandas as pd
from customer_cleaner import CustomerCleaner


In [2]:
old_df=pd.read_csv("customer_churn.csv")

In [3]:
cleaner=CustomerCleaner(old_df)
df=cleaner.clean()
df

2026-08-17 10:43:12,020 - INFO - CustomerCleaner initialized.
2026-08-17 10:43:12,023 - INFO - Input DataFrame Shape: (7043, 21)
2026-08-17 10:43:12,026 - INFO - Starting data cleaning pipeline...
2026-08-17 10:43:12,028 - INFO - Standardizing column names...
2026-08-17 10:43:12,033 - INFO - Column names converted to snake_case.
2026-08-17 10:43:12,036 - INFO - Cleaning total_charges column...
2026-08-17 10:43:12,048 - INFO - Blank strings found: 0
2026-08-17 10:43:12,056 - INFO - Conversion completed. Total null values in total_charges: 11
2026-08-17 10:43:12,059 - INFO - Normalizing binary columns...
2026-08-17 10:43:12,061 - INFO - Processing column: partner
2026-08-17 10:43:12,068 - INFO - Processing column: dependents
2026-08-17 10:43:12,072 - INFO - Processing column: phone_service
2026-08-17 10:43:12,077 - INFO - Processing column: paperless_billing
2026-08-17 10:43:12,081 - INFO - Processing column: churn
2026-08-17 10:43:12,084 - INFO - Binary columns normalized.
2026-08-17 10

,customer_i_d,gender,senior_citizen,partner,dependents,tenure,phone_service,multiple_lines,internet_service,online_security,...,device_protection,tech_support,streaming_t_v,streaming_movies,contract,paperless_billing,payment_method,monthly_charges,total_charges,churn
0,7590-VHVEG,Female,0,1,0,1,0,No phone service,DSL,No,...,No,No,No,No,Month-to-month,1,Electronic check,29.85,29.85,0
1,5575-GNVDE,Male,0,0,0,34,1,No,DSL,Yes,...,Yes,No,No,No,One year,0,Mailed check,56.95,1889.50,0
2,3668-QPYBK,Male,0,0,0,2,1,No,DSL,Yes,...,No,No,No,No,Month-to-month,1,Mailed check,53.85,108.15,1
3,7795-CFOCW,Male,0,0,0,45,0,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,0,Bank transfer (automatic),42.30,1840.75,0
4,9237-HQITU,Female,0,0,0,2,1,No,Fiber optic,No,...,No,No,No,No,Month-to-month,1,Electronic check,70.70,151.65,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7038,6840-RESVB,Male,0,1,1,24,1,Yes,DSL,Yes,...,Yes,Yes,Yes,Yes,One year,1,Mailed check,84.80,1990.50,0
7039,2234-XADUH,Female,0,1,1,72,1,Yes,Fiber optic,No,...,Yes,No,Yes,Yes,One year,1,Credit card (automatic),103.20,7362.90,0
7040,4801-JZAZL,Female,0,1,1,11,0,No phone service,DSL,Yes,...,No,No,No,No,Month-to-month,1,Electronic check,29.60,346.45,0
7041,8361-LTMKD,Male,1,1,0,4,1,Yes,Fiber optic,No,...,No,No,No,No,Month-to-month,1,Mailed check,74.40,306.60,1


27. Feature 1 — tenure_bucket: pd.cut() with bins [0,12,24,48,72] as in CP3 

In [4]:
df["tenure_bucket"]=pd.cut(
    df["tenure"],
    bins=[0,12,24,48,72],
    labels=["0-12", "13-24", "25-48", "49-72"],
    include_lowest=True
)
df

,customer_i_d,gender,senior_citizen,partner,dependents,tenure,phone_service,multiple_lines,internet_service,online_security,...,tech_support,streaming_t_v,streaming_movies,contract,paperless_billing,payment_method,monthly_charges,total_charges,churn,tenure_bucket
0,7590-VHVEG,Female,0,1,0,1,0,No phone service,DSL,No,...,No,No,No,Month-to-month,1,Electronic check,29.85,29.85,0,0-12
1,5575-GNVDE,Male,0,0,0,34,1,No,DSL,Yes,...,No,No,No,One year,0,Mailed check,56.95,1889.50,0,25-48
2,3668-QPYBK,Male,0,0,0,2,1,No,DSL,Yes,...,No,No,No,Month-to-month,1,Mailed check,53.85,108.15,1,0-12
3,7795-CFOCW,Male,0,0,0,45,0,No phone service,DSL,Yes,...,Yes,No,No,One year,0,Bank transfer (automatic),42.30,1840.75,0,25-48
4,9237-HQITU,Female,0,0,0,2,1,No,Fiber optic,No,...,No,No,No,Month-to-month,1,Electronic check,70.70,151.65,1,0-12
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7038,6840-RESVB,Male,0,1,1,24,1,Yes,DSL,Yes,...,Yes,Yes,Yes,One year,1,Mailed check,84.80,1990.50,0,13-24
7039,2234-XADUH,Female,0,1,1,72,1,Yes,Fiber optic,No,...,No,Yes,Yes,One year,1,Credit card (automatic),103.20,7362.90,0,49-72
7040,4801-JZAZL,Female,0,1,1,11,0,No phone service,DSL,Yes,...,No,No,No,Month-to-month,1,Electronic check,29.60,346.45,0,0-12
7041,8361-LTMKD,Male,1,1,0,4,1,Yes,Fiber optic,No,...,No,No,No,Month-to-month,1,Mailed check,74.40,306.60,1,0-12


28. Feature 2 — high_charge_flag: 1 if monthly_charges > median monthly_charges, else 0 

In [5]:
median_charge = df["monthly_charges"].median()
df["high_charge_flag"]=(
    df["monthly_charges"] > median_charge
).astype(int)


29. Feature 3 — service_count: count how many of these 6 service columns equal 1: online_security, online_backup, device_protection, tech_support, streaming_tv, streaming_movies 

In [6]:
service_cols = [
    "online_security",
    "online_backup",
    "device_protection",
    "tech_support",
    "streaming_t_v",      # notice your column name
    "streaming_movies"
]

for col in service_cols:
    df[col] = df[col].map({
        "Yes": 1,
        "No": 0,
        "No internet service": 0
    })

# Create service_count AFTER mapping
df["service_count"] = df[service_cols].sum(axis=1)
df["service_count"]

0       1
1       2
2       2
3       3
4       0
       ..
7038    5
7039    4
7040    1
7041    0
7042    5
Name: service_count, Length: 7043, dtype: int64

30. Feature 4 — is_long_term_customer: 1 if tenure >= 24, else 0 

In [7]:
df["is_long_term_customer"]=(df["tenure"]>=24).astype(int)
df["is_long_term_customer"]

0       0
1       1
2       0
3       1
4       0
       ..
7038    1
7039    1
7040    0
7041    0
7042    1
Name: is_long_term_customer, Length: 7043, dtype: int64

31. Feature 5 — has_streaming_bundle: 1 if both streaming_tv=1 AND streaming_movies=1, else 0 

In [8]:
df["has_streaming_bundle"] = (
    (df["streaming_t_v"] == 1) &
    (df["streaming_movies"] == 1)
).astype(int)
df["has_streaming_bundle"]

0       0
1       0
2       0
3       0
4       0
       ..
7038    1
7039    1
7040    0
7041    0
7042    1
Name: has_streaming_bundle, Length: 7043, dtype: int64

32. auto_pay_flag: 1 if payment_method contains "automatic", else 0 (use str.contains()) 

In [9]:
df["auto_pay_flag"] = (
    df["payment_method"]
      .str.contains("automatic")
      .astype(int)
)
df["auto_pay_flag"]

0       0
1       0
2       0
3       1
4       0
       ..
7038    0
7039    1
7040    0
7041    0
7042    1
Name: auto_pay_flag, Length: 7043, dtype: int64

33. Print the correlation of each new feature with the churn column 

In [10]:
new_features = [
    "high_charge_flag",
    "service_count",
    "is_long_term_customer",
    "has_streaming_bundle",
    "auto_pay_flag"
]

print(df[new_features + ["churn"]].corr()["churn"])

high_charge_flag         0.195663
service_count           -0.087698
is_long_term_customer   -0.311496
has_streaming_bundle     0.040441
auto_pay_flag           -0.209902
churn                    1.000000
Name: churn, dtype: float64


In [11]:
df.to_csv("customer_features.csv", index=False)

In [12]:
customer_features = pd.read_csv("customer_features.csv")
customer_features.head()

,customer_i_d,gender,senior_citizen,partner,dependents,tenure,phone_service,multiple_lines,internet_service,online_security,...,payment_method,monthly_charges,total_charges,churn,tenure_bucket,high_charge_flag,service_count,is_long_term_customer,has_streaming_bundle,auto_pay_flag
0,7590-VHVEG,Female,0,1,0,1,0,No phone service,DSL,0,...,Electronic check,29.85,29.85,0,0-12,0,1,0,0,0
1,5575-GNVDE,Male,0,0,0,34,1,No,DSL,1,...,Mailed check,56.95,1889.50,0,25-48,0,2,1,0,0
2,3668-QPYBK,Male,0,0,0,2,1,No,DSL,1,...,Mailed check,53.85,108.15,1,0-12,0,2,0,0,0
3,7795-CFOCW,Male,0,0,0,45,0,No phone service,DSL,1,...,Bank transfer (automatic),42.30,1840.75,0,25-48,0,3,1,0,1
4,9237-HQITU,Female,0,0,0,2,1,No,Fiber optic,0,...,Electronic check,70.70,151.65,1,0-12,1,0,0,0,0


In [14]:
df.dtypes

customer_i_d                  str
gender                        str
senior_citizen              int64
partner                     int64
dependents                  int64
tenure                      int64
phone_service               int64
multiple_lines                str
internet_service              str
online_security             int64
online_backup               int64
device_protection           int64
tech_support                int64
streaming_t_v               int64
streaming_movies            int64
contract                      str
paperless_billing           int64
payment_method                str
monthly_charges           float64
total_charges             float64
churn                       int64
tenure_bucket            category
high_charge_flag            int64
service_count               int64
is_long_term_customer       int64
has_streaming_bundle        int64
auto_pay_flag               int64
dtype: object